# Setup and orientation

Assistant version 0. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_agent_or_workflow

Source: [01_agent_or_workflow.py](examples/01_agent_or_workflow.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Model-driven agents versus fixed workflows; tokens; ecosystem map (Python, TypeScript, tools,
Evals, builder, Bedrock, AgentCore, MCP, A2A).

Run: uv run python modules/00_setup/examples/01_agent_or_workflow.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # A workflow always follows our branch; a model-driven agent chooses an answer.
    request = "Can Nimbus help me plan a trip?"
    workflow_answer = "Choose a destination." if "trip" in request else "Contact support."
    console.print("Workflow:", workflow_answer)
    # KEY IDEA: the Agent owns the model conversation; we give it a task.
    agent = Agent(
        model=get_model(cfg, responses=["Yes. Which destination and budget?"]),
        system_prompt="You are Nimbus. Ask for missing trip details.",
        callback_handler=None,
        retry_strategy=None,
    )
    complete(agent(request, limits={"turns": 2}), cfg)


run_lesson(main, "modules/00_setup/examples/01_agent_or_workflow.py")

## 02_local_readiness

Source: [02_local_readiness.py](examples/02_local_readiness.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Python, uv, pip/venv fallback, Ollama tool-capable model and configuration checks.

Run: uv run python modules/00_setup/examples/02_local_readiness.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import argparse
import sys

from common.config import load_settings
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from scripts.check_env import check_ollama


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--check-service", action="store_true")
    args, cfg = lesson_settings(parser)
    console.print("Python", sys.version.split()[0], "provider", cfg.provider)
    console.print("Free path: set MODEL_PROVIDER=ollama; leave MODEL_ID blank.")
    if args.check_service and not args.mock:
        check_ollama(load_settings(provider="ollama"))
    else:
        console.print("Offline configuration check; Ollama service not contacted.")
    # KEY IDEA: configuration readiness and service reachability are separate checks.
    assert get_model(provider="mock", responses=["ready"]).get_config()["model_id"] == "scripted"
    complete()


run_lesson(main, "modules/00_setup/examples/02_local_readiness.py")

## 03_aws_readiness

Source: [03_aws_readiness.py](examples/03_aws_readiness.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Identity Center, CLI v2 SSO, budget-before-billing gate, region and model-access checks.

Run: uv run python modules/00_setup/examples/03_aws_readiness.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Yes; local readiness instead. Resources created: none.
"""

import argparse

from common.lessons import complete, lesson_settings, require_bedrock
from common.pretty import console, run_cli
from scripts.check_env import check_aws


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--check-account", action="store_true")
    args, cfg = lesson_settings(parser)
    console.print("SSO -> named profile -> budget -> region/model access -> inference.")
    # KEY IDEA: the default explains readiness without contacting an AWS account.
    if args.check_account and not args.mock:
        require_bedrock(cfg)
        check_aws(cfg)
    else:
        console.print("Read docs/AWS_SETUP.md; no identity or model access has been verified.")
    complete()


run_lesson(main, "modules/00_setup/examples/03_aws_readiness.py")

## 04_first_call_metrics

Source: [04_first_call_metrics.py](examples/04_first_call_metrics.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""First configured agent call and token/latency inspection.

Run: uv run python modules/00_setup/examples/04_first_call_metrics.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    agent = Agent(
        model=get_model(cfg, responses=["Nimbus is ready to help."]),
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: metrics describe the actual loop, even with a scripted model.
    result = agent("Introduce Nimbus in one sentence.", limits={"turns": 2})
    complete(result, cfg)


run_lesson(main, "modules/00_setup/examples/04_first_call_metrics.py")

## orientation

Source: [orientation.py](use_case/orientation.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Orient a new Nimbus learner and measure one successful call.

Run: uv run python modules/00_setup/use_case/orientation.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    console.print("Nimbus learns in this order: model -> conversation -> structure -> tools.")
    console.print("Readiness checks: scripts/check_env.py; hosted setup: docs/AWS_SETUP.md")
    # KEY IDEA: first confirm a small end-to-end call before adding integrations.
    result = Agent(
        model=get_model(cfg, responses=["Nimbus is ready. What trip are you planning?"]),
        callback_handler=None,
        retry_strategy=None,
    )("Introduce Nimbus in one sentence.", limits={"turns": 2})
    complete(result, cfg)


run_lesson(main, "modules/00_setup/use_case/orientation.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
